# Fourier Pricing

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Purpose:** Demonstrate the Fourier-based option pricers exposed by `finstack_quant.valuations` and compare them to familiar Black-Scholes intuition.

**Prerequisites:** Basic option pricing, put-call parity, and the difference between diffusion and jump or pure-jump models.

**In this notebook:** We compare COS pricing under Black-Scholes, then extend the same framework to Variance Gamma and Merton jump-diffusion examples.


## Concept

Fourier pricers are useful because they let you evaluate several models through a similar interface once you know the characteristic function. Here we use them for three jobs:

1. Verify numerical agreement against Black-Scholes.
2. Compare smile behavior under a richer model.
3. Check parity and sanity conditions.


In [1]:
import math

import numpy as np

from finstack_quant.valuations import (
    bs_cos_price,
    bs_price,
    merton_jump_cos_price,
    vg_cos_price,
)


def check_parity(pricer, spot, strike, rate, dividend, maturity, *extra):
    """Put-call parity residual for a COS pricer: (C - P) - (S*e^{-qT} - K*e^{-rT})."""
    call = pricer(spot, strike, rate, dividend, *extra, maturity, True)
    put = pricer(spot, strike, rate, dividend, *extra, maturity, False)
    forward_disc = spot * math.exp(-dividend * maturity) - strike * math.exp(-rate * maturity)
    return call, put, (call - put) - forward_disc


## Model comparison

The next cell follows the original script closely: a Black-Scholes cross-check, a Variance Gamma smile, a Merton jump grid, and parity residuals. The goal is not just to print prices, but to show the kinds of invariants you would want to test whenever you add a new pricing method.


In [2]:
spot = 100.0
rate = 0.05
dividend = 0.02
maturity = 1.0
bs_vol = 0.20
strike_atm = 100.0

cf_price = bs_price(spot, strike_atm, rate, dividend, bs_vol, maturity, True)
cos_price = bs_cos_price(spot, strike_atm, rate, dividend, bs_vol, maturity, True)
print("Black-Scholes ATM call")
print(f"  closed form : {cf_price:.6f}")
print(f"  COS         : {cos_price:.6f}")
print(f"  COS error   : {abs(cos_price - cf_price):.2e}")

strikes = np.array([80.0, 90.0, 95.0, 100.0, 105.0, 110.0, 120.0])
vg_sigma, vg_theta, vg_nu = 0.20, -0.10, 0.20
m_sigma, m_lambda, m_mu_j, m_sigma_j = 0.20, 0.50, -0.10, 0.15

vg_calls = np.array([
    vg_cos_price(spot, float(k), rate, dividend, vg_sigma, vg_theta, vg_nu, maturity, True)
    for k in strikes
])

merton_calls = np.array([
    merton_jump_cos_price(
        spot,
        float(k),
        rate,
        dividend,
        m_sigma,
        m_mu_j,
        m_sigma_j,
        m_lambda,
        maturity,
        True,
    )
    for k in strikes
])

print("\nVariance Gamma call smile")
for strike, price in zip(strikes, vg_calls):
    print(f"  K={strike:>6.2f} -> {price:.6f}")

print("\nMerton jump-diffusion call grid")
for strike, price in zip(strikes, merton_calls):
    print(f"  K={strike:>6.2f} -> {price:.6f}")

parity_strike = 105.0
for label, pricer, extra in (
    ("BS/COS", bs_cos_price, (bs_vol,)),
    ("VG/COS", vg_cos_price, (vg_sigma, vg_theta, vg_nu)),
    ("Merton/COS", merton_jump_cos_price, (m_sigma, m_mu_j, m_sigma_j, m_lambda)),
):
    _, _, residual = check_parity(pricer, spot, parity_strike, rate, dividend, maturity, *extra)
    print(f"\n{label:<12} parity residual: {residual:.2e}")


Black-Scholes ATM call
  closed form : 9.227006
  COS         : 9.227006
  COS error   : 2.49e-14

Variance Gamma call smile
  K= 80.00 -> 22.978627
  K= 90.00 -> 15.267119
  K= 95.00 -> 11.992746
  K=100.00 -> 9.182529
  K=105.00 -> 6.859085
  K=110.00 -> 5.009916
  K=120.00 -> 2.535071

Merton jump-diffusion call grid
  K= 80.00 -> 23.521847
  K= 90.00 -> 16.202867
  K= 95.00 -> 13.103613
  K=100.00 -> 10.416477
  K=105.00 -> 8.143365
  K=110.00 -> 6.265938
  K=120.00 -> 3.551974

BS/COS       parity residual: 1.78e-14

VG/COS       parity residual: 1.78e-14

Merton/COS   parity residual: 1.78e-14


## Takeaways

- `bs_cos_price()` is a natural regression check: it should agree with the analytic `bs_price()` closed form to numerical tolerance.
- `vg_cos_price()` and `merton_jump_cos_price()` show how richer distributions change strike-by-strike prices without changing the overall workflow.
- Parity checks are cheap and valuable whenever you build a new pricing notebook or add a new model binding.


In [3]:
{
    "bs_closed_form": round(cf_price, 6),
    "bs_cos": round(cos_price, 6),
    "vg_call_min": round(float(vg_calls.min()), 6),
    "merton_call_max": round(float(merton_calls.max()), 6),
}


{'bs_closed_form': 9.227006,
 'bs_cos': 9.227006,
 'vg_call_min': 2.535071,
 'merton_call_max': 23.521847}

## Closed-form primitives (BS + exotics)

Alongside `bs_price` (used above as the closed-form reference), the root module exposes closed-form exotic pricers for quick checks — independent of the full instrument JSON pipeline.


In [4]:
from finstack_quant.valuations import asian_option_price, barrier_call

print("BS call:", round(bs_price(100, 100, 0.05, 0.01, 0.20, 1.0, True), 6))
print("Asian arith:", round(asian_option_price(100, 100, 0.05, 0.0, 0.20, 1.0, 4, "arithmetic", True), 6))
print("Barrier down-out:", round(barrier_call(100, 100, 90, 0.05, 0.0, 0.20, 1.0, "down", "out"), 6))


BS call: 9.826298
Asian arith: 6.95424
Barrier down-out: 8.665472
